# AI Gaming Editor - Colab Demo\n\nتجريب مشروع تحليل فيديو الـ Gaming وتحويله لريلز احترافية\n\n**المتطلبات:**\n- حساب Google Drive فيه مجلد `videos` يحتوي على فيديو gaming\n- اختيار GPU Runtime من Colab (T4 مثلاً)\n\n**الخطوات:**\n1. الاتصال بـ Google Drive\n2. تثبيت المكتبات المطلوبة\n3. استنساخ المشروع\n4. اختيار فيديو من Drive\n5. تشغيل Pipeline التحليل\n6. عرض النتائج وتحميل الريل

## 1. الاتصال بـ Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. التحقق من GPU

In [ ]:
import torch
import os

!nvidia-smi
print(f"\nPyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. تثبيت المكتبات المطلوبة

In [ ]:
# 1. تحديث النظام ليتوافق مع بايثون 3.12 وأحدث حزم المونتاج والـ LLMs
!pip install -U -q pip setuptools wheel
!pip install -U -q numpy

# 2. تثبيت مكتبات معالجة الصوت والرؤية الحاسوبية وقراءة الـ Kill Feed
!pip install -q opencv-python-headless librosa scenedetect[opencv] ultralytics openai easyocr numba

# 3. تثبيت عائلة Whisper للتعرف على الكلام والصوت
!pip install -q faster-whisper
!pip install -q git+https://github.com/m-bain/whisperx.git

## 4. استنساخ المشروع

In [ ]:
import os
import shutil

%cd /content

if os.path.exists("ai_gaming_editor"):
    print("Cleaning old repository...")
    shutil.rmtree("ai_gaming_editor")

!git clone https://github.com/sadek-marouf/ai_gaming_editor.git

%cd ai_gaming_editor
print("\nProject files configuration:")
!ls -la

## 5. اختيار فيديو من Google Drive\n\nتأكد أن عندك مجلد `videos` في الـ Drive يحتوي على فيديو gaming.\nالخلية التالية تعرض الفيديوهات المتوفرة وتخليك تختار واحد.

In [ ]:
import os

DRIVE_VIDEOS_DIR = "/content/drive/MyDrive/videos"
LOCAL_VIDEO_DIR = "/content/ai_gaming_editor/input_videos"

os.makedirs(LOCAL_VIDEO_DIR, exist_ok=True)
video_extensions = ('.mp4', '.avi', '.mkv', '.mov', '.webm', '.flv')

if not os.path.exists(DRIVE_VIDEOS_DIR):
    print(f"ERROR: '{DRIVE_VIDEOS_DIR}' not found!")
    print("Please create a 'videos' folder in your Google Drive root and add gaming videos to it.")
else:
    videos = [f for f in os.listdir(DRIVE_VIDEOS_DIR) if f.lower().endswith(video_extensions)]

    if not videos:
        print(f"No video files found in {DRIVE_VIDEOS_DIR}")
        print(f"Supported formats: {video_extensions}")
    else:
        print(f"Found {len(videos)} video(s) in Drive/videos:\n")
        for i, v in enumerate(videos):
            size_mb = os.path.getsize(os.path.join(DRIVE_VIDEOS_DIR, v)) / (1024 * 1024)
            print(f"  [{i}] {v} ({size_mb:.1f} MB)")

## 6. نسخ الفيديو المختار\n\nغيّر `VIDEO_INDEX` لاختيار فيديو مختلف من القائمة أعلاه.

In [ ]:
import os
import shutil

# ==============================
# اختر رقم الفيديو من القائمة أعلاه
# ==============================
VIDEO_INDEX = 0

if 'videos' in locals() and videos:
    base_name = os.path.splitext(videos[VIDEO_INDEX])[0]
    src = os.path.join(DRIVE_VIDEOS_DIR, videos[VIDEO_INDEX])
    dst = os.path.join(LOCAL_VIDEO_DIR, videos[VIDEO_INDEX])

    print(f"Copying: {videos[VIDEO_INDEX]}...")
    shutil.copy2(src, dst)

    VIDEO_PATH = dst
    print(f"Video ready: {VIDEO_PATH}")
    print(f"Size: {os.path.getsize(VIDEO_PATH) / (1024*1024):.1f} MB")
else:
    print("ERROR: Run cell 5 first to read videos list.")

## 7. معلومات الفيديو

In [ ]:
import cv2

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = total_frames / fps if fps > 0 else 0
cap.release()

print(f"Resolution: {width}x{height}")
print(f"FPS: {fps}")
print(f"Total Frames: {total_frames}")
print(f"Duration: {duration:.1f} sec ({duration/60:.1f} min)")

## 8. إعداد Groq AI (اختياري)\n\nإذا عندك مفتاح Groq API، فعّله هنا للحصول على تقييم ذكاء اصطناعي للمقاطع.\nإذا ما عندك، المشروع يشتغل بدونه بشكل طبيعي.

In [ ]:
import os
from google.colab import userdata

# يفضّل وضع المفتاح في خانة Secrets باسم GROQ_API_KEY
try:
    GROQ_API_KEY = userdata.get('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')
except:
    GROQ_API_KEY = "" # أو ضعه هنا مباشرة إذا أردت: "gsk_xxx"

if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("✅ Groq AI ENABLED")
else:
    print("⚠️ Groq AI DISABLED (Optional - script will use offline logic)")

## 9. اختيار اللعبة وتشغيل Pipeline

اختر بروفايل اللعبة من القائمة المتوفرة:
- `pubg` — PUBG / PUBG Mobile (تخصيص كامل: kill feed, hitmarker, فلترة مركبات, تأثيرات بصرية)
- `generic` — عام (يعمل مع أي لعبة بدون تأثيرات)

**ميزات PUBG الجديدة:**
- **توقيت ذكي**: يبدأ المقطع 3 ثوانٍ قبل لحظة القتل ويستمر 2-3 ثوانٍ بعدها
- **فلاش بصري**: تعزيز contrast/saturation لمدة 0.2 ثانية عند لحظة القتل
- **سلو موشن**: 0.5x لمدة 1.5 ثانية بعد القتل مع صوت pitch-corrected
- **صوت انتقالي**: swoosh sub-bass عند نهاية كل مقطع

In [ ]:
import os

GAME = "pubg"        # البروفايل المدعوم بالتوقيت الذكي والمونتاج الاحترافي
QUALITY = "medium"   # low / medium / high
WORKERS = 4          # عدد العمليات المتوازية
OUTPUT_DIR = "/content/ai_gaming_editor/outputs"

print(f"🚀 Starting AI Gaming Editor Core for Profile: {GAME}")
print(f"🎥 Targeting Video Path: {VIDEO_PATH}")
print("="*60 + "\n")

# حقن بيئة النظام بتعطيل تحذيرات Numba والتعارضات للعمل غصب عنها بكفاءة
os.environ["NUMBA_DISABLE_INTEL_SVML"] = "1"

# تشغيل عملية النظام المستقلة لتطبيق التوقيت والمونتاج والقطع والفلاش البصري والسلو موشن
!PYTHONWARNINGS="ignore" python /content/ai_gaming_editor/main.py \
    --game "$GAME" \
    --out "$OUTPUT_DIR" \
    --quality "$QUALITY" \
    --workers $WORKERS \
    "$VIDEO_PATH"

# فحص العثور على الريل الناتج عالي الجودة والمنقح
base_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
expected_reel = os.path.join(OUTPUT_DIR, base_name, "reels", "viral_reel.mp4")

if os.path.exists(expected_reel):
    result = expected_reel
    print("\n" + "="*60)
    print(f"🎉 SUCCESS! Professional Reel generated: {result}")
    print("="*60)
else:
    result = None
    print("\n" + "="*60)
    print("❌ FAILED: Processing complete but reel asset wasn't created. Check logs above.")
    print("="*60)

## 10. عرض نتائج التحليل

In [ ]:
import json
import os

segments_path = os.path.join(OUTPUT_DIR, base_name, "segments.json")

if os.path.exists(segments_path):
    with open(segments_path, "r", encoding="utf-8") as f:
        segments = json.load(f)

    print(f"Top {len(segments)} segments selected for reel with Effects Engine:\n")
    print(f"{'#':<4} {'Start':>8} {'End':>8} {'Score':>8} {'Audio':>8} {'Kill':>8} {'Triggers':>10} Text")
    print("-" * 100)

    for i, seg in enumerate(segments, 1):
        text_preview = seg['text'].replace('\N', ' ')[:35]
        triggers = seg.get('triggers', [])
        trigger_types = ", ".join(set(t['type'] for t in triggers)) if triggers else "-"
        print(
            f"{i:<4} {seg['start']:>8.1f} {seg['end']:>8.1f} "
            f"{seg['score']:>8.3f} {seg.get('audio', 0):>8.3f} "
            f"{seg.get('kill_feed', 0):>8.3f} {trigger_types:>10} "
            f"{text_preview}"
        )

    has_triggers = any(seg.get('triggers') for seg in segments)
    if has_triggers:
        print(f"\nTrigger & Flash Sync details:")
        for i, seg in enumerate(segments, 1):
            for t in seg.get('triggers', []):
                print(f"  Seg {i}: Dynamic {t['type']} applied at {t['time']}s (strength={t['strength']:.2f})")
else:
    print("segments.json not found - pipeline execution might have failed")

## 11. عرض الريل في Colab

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

if 'result' in locals() and result and os.path.exists(result):
    reel_size = os.path.getsize(result) / (1024 * 1024)
    print(f"Reel size: {reel_size:.1f} MB")

    if reel_size < 50:
        with open(result, "rb") as f:
            video_data = f.read()
        video_b64 = b64encode(video_data).decode()
        display(HTML(f"""
        <video width="360" height="640" controls>
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        </video>
        """))
    else:
        print(f"Video too large to display inline ({reel_size:.0f} MB). Download from left sidebar: {result}")
else:
    print("No rendered reel found to display.")

## 12. حفظ الريل على Google Drive

In [ ]:
import os
import shutil

DRIVE_OUTPUT = "/content/drive/MyDrive/videos/reels"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

if 'result' in locals() and result and os.path.exists(result):
    output_name = f"{base_name}_pro_reel.mp4"
    drive_path = os.path.join(DRIVE_OUTPUT, output_name)
    shutil.copy2(result, drive_path)
    print(f"🔥 Success! Pro Reel saved to Drive: {drive_path}")

    segments_path = os.path.join(OUTPUT_DIR, base_name, "segments.json")
    if os.path.exists(segments_path):
        analysis_dest = os.path.join(DRIVE_OUTPUT, f"{base_name}_segments.json")
        shutil.copy2(segments_path, analysis_dest)
        print(f"Analysis logs backup saved to Drive: {analysis_dest}")
else:
    print("No valid reel available for saving.")

## 13. تنظيف الملفات المؤقتة (اختياري)

In [ ]:
# uncomment to clean up\n# !rm -rf /content/ai_gaming_editor/outputs\n# !rm -rf /content/ai_gaming_editor/input_videos\n\nprint("Done! Check your Drive/videos/reels folder for the output.")